In [10]:
import os

from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel
from dotenv import load_dotenv
from openai import AsyncOpenAI


In [2]:
load_dotenv(override=True)

True

In [31]:
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek_base_url = os.getenv("DEEPSEEK_BASE_URL")
deepseek_model = os.getenv("DEEPSEEK_MODEL")

token_harbor_api_key = os.getenv("TOKEN_HARBOR_API_KEY")
token_harbor_base_url = os.getenv("TOKEN_HARBOR_BASE_URL")
token_harbor_model = os.getenv("TOKEN_HARBOR_MODEL")

In [5]:
instruction = """
You are a sales agent working for Finance,
a company that provides a SaaS tool for ensuring compliance with the GDPR.
You write compelling emails to customers to help them comply with the GDPR.
"""

In [32]:
deepseek_client = AsyncOpenAI(
    api_key=deepseek_api_key,
    base_url=deepseek_base_url
)

harbor_client = AsyncOpenAI(
    api_key=token_harbor_api_key,
    base_url=token_harbor_base_url
)

In [34]:
deepseek_chat_model = OpenAIChatCompletionsModel(model=deepseek_model, openai_client=deepseek_client)
harbor_chat_model = OpenAIChatCompletionsModel(model=token_harbor_model, openai_client=harbor_client)

In [35]:
sales_agent1 = Agent(name="SalesAgent1", instructions=instruction, model=deepseek_chat_model)
sales_agent2 = Agent(name="SalesAgent2", instructions=instruction, model=harbor_chat_model)
sales_agent3 = Agent(name="SalesAgent3", instructions=instruction, model=harbor_chat_model)

In [36]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [14]:
from messeger import send_email, push


def send_message(subject, text_body, html_body):
    send_email(subject, text_body, html_body)
    push(text_body)

In [16]:
send_message("Yet another test", "This is a test", "<p>This is a test</p>")

Pushing message to Pushover: This is a test


In [17]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """Send an email with the given subject and body to the specified recipient."""
    send_email(subject, text_body, html_body)
    return f"Email sent to {os.getenv('EMAIL_RECIPIENT')}"

In [18]:
tools = [tool1, tool2, tool3, send_email_tool]

In [37]:
instruction = """
You are a Sales Manager at ComplAI. your gold is to find the single cold sales email using the sales_writer tools.
"""

tasks = """
Follow these steps

1. Generate Drafts: use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write s sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and select: review the drafts and choose the single best email using your judgement of which one is most effective.

3. Use your tool to send the best email (and only the best email) to the use. only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instruction, model=deepseek_chat_model, tools=tools)


In [20]:
with trace("Sales Manager across different model"):
    result = await Runner.run(sales_manager, tasks)
print(result.final_output)

Done. Here's a summary of the workflow:

**1. Generated 3 drafts** — one from each sales writer tool (all independent, run in parallel).

**2. Evaluated & selected** — I chose **Draft 3 (sales_agent3)** as the most effective. It had the strongest hook, concrete and vivid benefits, a clear CTA with an added-value offer (free GDPR readiness checklist), a low-pressure P.S., and a professional unsubscribe footer.

**3. Sent exactly one email** — Draft 3 was delivered successfully to hieu.nguyenvan.dev10@gmail.com.

All steps completed with only a single email sent, as instructed.


In [21]:
from pydantic import BaseModel, Field


class EmailReview(BaseModel):
    is_professional: bool = Field(description="Indicates whether the email is professional in tone and content.")
    number_of_sentences: int = Field(description="The total number of sentences in the email.")
    contains_placeholders: bool = Field(
        description="Indicates whether the email contains placeholders that need to be filled in.")

In [22]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Indicates whether the email is professional in tone and content.',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The total number of sentences in the email.',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Indicates whether the email contains placeholders that need to be filled in.',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [24]:
email = """
Hi [first_name]
I am writing to you to let you know that I am available for hire.
laters.
Regards,
[last_name]
"""

In [26]:
checker = Agent(name="checker", instructions="You review potential sales emails", model="gpt-5-nano",
                output_type=EmailReview)
result = await Runner.run(checker, email)

In [28]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=2, contains_placeholders=True)

In [38]:
deepseek_agent = Agent(name="checker", instructions="You review potential sales emails", model=deepseek_chat_model,
                       output_type=EmailReview)
result = await Runner.run(deepseek_agent, email)

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now (request_id: 99ed2c7f-a4bc-428d-8aab-e89c58f2f553)', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}